# Notebook 03 — Modelo XGBoost
**Entrenamiento · Comparación · Diagnóstico**

In [ ]:
!pip install xgboost scikit-learn imbalanced-learn matplotlib seaborn -q
print('✓ Listo')

In [ ]:
import pandas as pd, numpy as np, pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, f1_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings; warnings.filterwarnings('ignore')

C1='#E63946'; C2='#457B9D'; C3='#2D6A4F'; C4='#F4A261'
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']

from google.colab import files
uploaded = files.upload()  # sube diabetes_procesado.csv
import io
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
X = df[features].values; y = df['Outcome'].values
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
smote = SMOTE(random_state=42)
X_train_sm,y_train_sm = smote.fit_resample(X_train_sc,y_train)
print(f'✓ Datos preparados: train={len(X_train_sm)}, test={len(X_test)}')

In [ ]:
# Entrenar modelos
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    min_child_weight=3, gamma=0.1,
    eval_metric='logloss', random_state=42, verbosity=0)
xgb_model.fit(X_train_sm, y_train_sm,
              eval_set=[(X_test_sc, y_test)], verbose=False)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sm, y_train_sm)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sm, y_train_sm)

y_pred  = xgb_model.predict(X_test_sc)
y_proba = xgb_model.predict_proba(X_test_sc)[:,1]
roc_auc = roc_auc_score(y_test, y_proba)
f1      = f1_score(y_test, y_pred)
lr_auc  = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1])
rf_auc  = roc_auc_score(y_test, rf.predict_proba(X_test_sc)[:,1])

print(f'\n✓ RESULTADOS:')
print(f'  Logistic Regression: ROC-AUC={lr_auc:.4f}')
print(f'  Random Forest:       ROC-AUC={rf_auc:.4f}')
print(f'  XGBoost:             ROC-AUC={roc_auc:.4f} ← seleccionado')
print(f'  XGBoost F1:          {f1:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Sin diabetes","Con diabetes"])}')

In [ ]:
# Validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train_sm, y_train_sm,
                             cv=cv, scoring='roc_auc')
print(f'CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Scores por fold: {cv_scores.round(4)}')

In [ ]:
# Figura 6 — Comparación de modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
for modelo_obj, nombre, color in [
        (lr,'Logistic Regression',C4),
        (rf,'Random Forest',C2),
        (xgb_model,'XGBoost ★',C1)]:
    proba = modelo_obj.predict_proba(X_test_sc)[:,1]
    fpr,tpr,_ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    lw = 3 if 'XGBoost' in nombre else 1.5
    ax.plot(fpr, tpr, lw=lw, color=color, label=f'{nombre} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Aleatorio (AUC=0.500)')
ax.set_xlabel('Tasa Falsos Positivos'); ax.set_ylabel('Tasa Verdaderos Positivos')
ax.set_title('Curvas ROC', fontweight='bold'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax2 = axes[1]
modelos_n = ['Logistic\nRegression','Random\nForest','XGBoost\n★']
aucs = [lr_auc, rf_auc, roc_auc]
bars = ax2.bar(modelos_n, aucs, color=[C4,C2,C1], width=0.5, alpha=0.9)
bars[2].set_edgecolor('black'); bars[2].set_linewidth(2)
for bar,val in zip(bars,aucs):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylim(0.5,1.0); ax2.set_ylabel('ROC-AUC')
ax2.set_title('Comparación ROC-AUC', fontweight='bold'); ax2.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show(); print('✓ Figura 6 guardada')

In [ ]:
# Figura 7 — Diagnóstico
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Sin diabetes','Con diabetes'],
            yticklabels=['Sin diabetes','Con diabetes'],
            linewidths=0.5, cbar=False, annot_kws={'size':14})
axes[0].set_title(f'Matriz de Confusión\nXGBoost', fontweight='bold')
axes[0].set_xlabel('Predicción'); axes[0].set_ylabel('Real')

prec,rec,_ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1].plot(rec, prec, color=C1, lw=2.5, label=f'XGBoost (AP={ap:.3f})')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', lw=1.2, label='Baseline')
axes[1].fill_between(rec, prec, alpha=0.1, color=C1)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].hist(y_proba[y_test==0], bins=25, alpha=0.6, color=C2, label='Sin diabetes', density=True)
axes[2].hist(y_proba[y_test==1], bins=25, alpha=0.6, color=C1, label='Con diabetes', density=True)
axes[2].axvline(0.5, color='black', linestyle='--', lw=1.5, label='Umbral (0.5)')
axes[2].set_xlabel('Probabilidad predicha'); axes[2].set_ylabel('Densidad')
axes[2].set_title('Distribución de probabilidades', fontweight='bold')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
plt.suptitle(f'Diagnóstico XGBoost — ROC-AUC={roc_auc:.4f} | F1={f1:.4f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('07_diagnostico_modelo.png', dpi=150, bbox_inches='tight')
plt.show(); print('✓ Figura 7 guardada')

In [ ]:
# Guardar modelo
with open('xgboost_diabetes.pkl','wb') as f:
    pickle.dump({'model':xgb_model,'scaler':scaler,'features':features}, f)
print('✓ Modelo guardado: xgboost_diabetes.pkl')
from google.colab import files
files.download('xgboost_diabetes.pkl')
for img in ['06_comparacion_modelos.png','07_diagnostico_modelo.png']:
    files.download(img)
print('✓ Archivos descargados')